# Measurement-bounded autonomy — interactive demo (Paper 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lesiayanytska78/ci-monitoring-simulation/blob/main/paper3_agentic_governance/notebooks/paper3_governance_demo.ipynb)

**The principle:** an autonomous energy agent may act *only where its sensing layer's
characterized detection performance says it can see* — and must escalate to a human where
the sensor is characterized as blind. This notebook lets you **drag a fault around the
(onset, severity) plane and watch the agent's authority switch on and off** at the measured
detection boundary, all driven by the real archived detection surface released with the paper.

Run **Runtime ▸ Run all**, then play with the sliders. Nothing to install by hand.

> Companion to the Paper 2 detector demo (`notebooks/quickstart.ipynb`); that one shows *how
> the detector sees*, this one shows *how the governance layer decides what the agent may do*.

## 0 · Setup — fetch the released artifact (no install)

Clones the published `ci-monitoring-simulation` repo (v3.0.0) and enters the Paper 3 folder.
Idempotent: safe to re-run.

In [ ]:
import os, sys, subprocess

def _sh(*a): subprocess.run(list(a), check=True)

# Find or fetch the Paper 3 package folder.
CAND = [os.path.join(os.getcwd(), "contract", "surface.py"),
        "ci-monitoring-simulation/paper3_agentic_governance/contract/surface.py"]
if os.path.exists(CAND[0]):
    PKG = os.getcwd()                                   # already inside the folder
else:
    if not os.path.isdir("ci-monitoring-simulation"):
        _sh("git", "clone", "--depth", "1",
            "https://github.com/lesiayanytska78/ci-monitoring-simulation.git")
    PKG = os.path.abspath("ci-monitoring-simulation/paper3_agentic_governance")

os.chdir(PKG); sys.path.insert(0, PKG)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyyaml", "ipywidgets", "matplotlib", "numpy", "pandas"])
print("Paper 3 package:", PKG)

In [ ]:
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
from contract.surface import DetectabilitySurface
from contract.governance import (EpistemicContract, ACTIONS, AUTONOMY_LABEL,
                                  Autonomy, Sigma, sigma_from_pdet, autonomy_grant)

SURF = DetectabilitySurface()
C    = EpistemicContract()
REG  = {a["name"]: a for a in yaml.safe_load(open("contract/fault_registry.yaml"))["archetypes"]}

SIGMA_NAME = {0: "BLIND", 1: "DEGRADED", 2: "TRANSITION", 3: "RELIABLE"}
TIER_COLORS = ["#dcdcdc", "#e8a87c", "#f4d58d", "#bdd7a3"]   # blind→reliable
print("σ cut-points:", C.cuts)
print(C.cuts_source)

## 1 · What the sensor can and cannot see

The contract reads a **characterized detectability surface** `P_det(s, ρ)` for each detector,
evaluated at the conservative **95% lower confidence bound**. `s = Δ/P_b` is fault severity
(fraction of baseline power); `ρ = τ_onset/W_b` is how slow the fault is relative to the
monitoring window. Below: the deployed rolling-baseline detector **D0** vs the event-anchored
**D2**. Bright = the sensor reliably sees a fault there; dark = it doesn't.

In [ ]:
rhos = np.linspace(0.05, 2.0, 160)
sevs = np.linspace(0.10, 1.20, 160)

def surface_grid(det, field="lo95"):
    Z = np.full((len(sevs), len(rhos)), np.nan)
    for i, s in enumerate(sevs):
        for j, r in enumerate(rhos):
            d = SURF.query(s, r, det)
            if d.in_envelope:
                Z[i, j] = d.p_det_lo95 if field == "lo95" else d.p_det
    return Z

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
for ax, det in zip(axes, ("D0", "D2")):
    Z = surface_grid(det)
    pc = ax.pcolormesh(rhos, sevs, Z, cmap="viridis", vmin=0, vmax=1, shading="auto")
    ax.set_xscale("log"); ax.set_title(f"{det}: P_det (95% lower bound)")
    ax.set_xlabel("onset ratio  ρ = τ_onset / W_b  (log)")
axes[0].set_ylabel("severity  s = Δ / P_b")
fig.colorbar(pc, ax=axes, label="P_det (lo95)", shrink=0.9)
plt.show()
print("D2 sees slow-onset, lower-severity faults that D0 (rolling baseline) drifts along with and misses.")

## 2 · The autonomy map — drag a fault and watch the decision  ⭐

This is the core of the framework. Move the sliders to place a fault (severity `s`, onset `ρ`),
pick the **detector** and the **action** the agent wants to take. The map recolours into the four
**sensing tiers** (σ), the ✱ marks your fault, and the panel underneath shows the *entire decision
being made*: the measured detectability → sensing tier σ → action-reversibility tier α →
**grant A = min(σ, α)** → the verdict.

Watch the coloured region — the agent's autonomy — **expand when you switch D0 → D2**. The gate
never changes; the *instrument* does.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def sigma_grid(det):
    G = np.zeros((len(sevs), len(rhos)), int)
    for i, s in enumerate(sevs):
        for j, r in enumerate(rhos):
            d = SURF.query(s, r, det)
            G[i, j] = int(sigma_from_pdet(d.p_det_lo95, C.cuts)) if d.in_envelope else 0
    return G

from matplotlib.colors import ListedColormap, BoundaryNorm
CMAP = ListedColormap(TIER_COLORS); NORM = BoundaryNorm([-.5,.5,1.5,2.5,3.5], CMAP.N)
_SG = {d: sigma_grid(d) for d in ("D0", "D2")}

def decide(s, rho, det, action_name):
    d = SURF.query(s, rho, det)
    gd = C.gate_action(action_name, s_hat=s, rho_hat=rho, detector=det)
    blind_kind = None
    if gd.sigma == Sigma.BLIND:
        blind_kind = "characterized-blind (in envelope, P_det too low)" if d.in_envelope \
                     else "MEASUREMENT-DEMAND (below the measured floor → order a characterization)"
    return d, gd, blind_kind

def show(severity=0.35, onset_rho=0.6, detector="D0", action="defer_job"):
    fig, ax = plt.subplots(figsize=(8.2, 5.2))
    ax.pcolormesh(rhos, sevs, _SG[detector], cmap=CMAP, norm=NORM, shading="auto")
    ax.set_xscale("log")
    ax.scatter([onset_rho], [severity], marker="*", s=420, c="#c0143c",
               edgecolor="white", linewidth=1.4, zorder=6)
    for nm, mk in [("compressed_air_leak","o"),("coolant_pump_fault","^"),
                   ("machine_left_on","s"),("tool_wear","x")]:
        fc = REG[nm]
        ax.scatter(fc["onset_ratio_nominal"], fc["severity_frac_nominal"],
                   c="#222", marker=mk, s=46, zorder=5)
    ax.set_xlabel("onset ratio  ρ  (log)"); ax.set_ylabel("severity  s = Δ / P_b")
    ax.set_title(f"Autonomy map — {detector}   (✱ = your fault)")
    handles = [plt.Rectangle((0,0),1,1, fc=c) for c in TIER_COLORS]
    ax.legend(handles, ["σ=0 blind","σ=1 degraded","σ=2 transition","σ=3 reliable"],
              loc="upper right", fontsize=8, framealpha=.9)
    plt.show()

    d, gd, blind_kind = decide(severity, onset_rho, detector, action)
    a = ACTIONS[action]
    print(f"  fault           s={severity:.2f}, ρ={onset_rho:.2f}   detector={detector}")
    print(f"  P_det (lo95)    {d.p_det_lo95:.3f}   [point estimate {d.p_det:.3f}]"
          + ("" if d.in_envelope else "   ← OUTSIDE measured envelope"))
    print(f"  sensing tier σ  {int(gd.sigma)}  ({SIGMA_NAME[int(gd.sigma)]})")
    print(f"  action '{action}'  α={a.alpha}  (ISA-95 L{a.isa95_level}"
          + (", request-only cap" if a.isa95_level == 2 else "") + ")")
    print(f"  GRANT  A=min(σ,α)  →  {AUTONOMY_LABEL[gd.autonomy]}")
    if blind_kind: print(f"  blind kind      {blind_kind}")
    print(f"  verdict         {'ESCALATE / propose' if gd.escalated else 'AUTONOMOUS action permitted'}")

interact(show,
         severity=widgets.FloatSlider(min=0.10, max=1.20, step=0.01, value=0.35, description="severity s"),
         onset_rho=widgets.FloatSlider(min=0.05, max=2.0, step=0.01, value=0.60, description="onset ρ"),
         detector=widgets.ToggleButtons(options=["D0","D2"], description="detector"),
         action=widgets.Dropdown(options=list(ACTIONS.keys()), value="defer_job", description="action"));

## 3 · The four registry archetypes, D0 → D2

The same trace, run for the four fault archetypes in the registry (severities from the
peer-reviewed characterization). Note **coolant** flipping from *degraded* (propose-and-wait)
under D0 to *reliable* (autonomous) under D2 — a recovered class — while **machine-left-on**
and **tool-wear** stay blind as *measurement-demands* under both (they sit below the measured
floor, so the contract refuses an unmeasured grant).

In [ ]:
rows = []
for name, fc in REG.items():
    r = {"archetype": name, "s": fc["severity_frac_nominal"], "ρ": fc["onset_ratio_nominal"]}
    for det in ("D0", "D2"):
        g = C.standing_grant(fc, "defer_job", det)
        tag = SIGMA_NAME[int(g.sigma)]
        if g.situation.get("blind_kind") == "measurement_demand": tag += " (meas-demand)"
        r[f"{det} σ"] = tag
        r[f"{det} grant"] = AUTONOMY_LABEL[g.autonomy].split(" ", 1)[0]
    rows.append(r)
pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

## 4 · Why the gate matters — three policies, live

Now the payoff. We run the Monte-Carlo policy matrix (naive always-autonomous / governance-gated
/ human-only) over the two detectors and four archetypes, on the real detection probabilities.
The **wrong-action rate** (acting, or optimizing, on a corrupted signal) is what the governance
layer eliminates where the sensor is weak.

In [ ]:
from experiments.run_policies import run
res = pd.DataFrame(run())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.3))
POLS, DETS = ["naive","gated","human"], ["D0","D2"]; w = 0.35
for k, det in enumerate(DETS):
    vals = [res[(res.detector==det)&(res.policy==p)].wrong_rate.iloc[0] for p in POLS]
    ax1.bar(np.arange(len(POLS))+k*w, vals, w, label=det)
    for x, v in zip(np.arange(len(POLS))+k*w, vals):
        ax1.text(x, v+0.015, f"{v:.2f}", ha="center", fontsize=8)
ax1.set_xticks(np.arange(len(POLS))+w/2); ax1.set_xticklabels(POLS)
ax1.set_ylim(0, .8); ax1.set_title("wrong-action rate"); ax1.legend(title="detector")

for k, pol in enumerate(POLS):
    em = [res[(res.detector==d)&(res.policy==pol)].emissions_kg.iloc[0] for d in DETS]
    ax2.bar(np.arange(len(DETS))+k*0.25, em, 0.25, label=pol)
ax2.set_xticks(np.arange(len(DETS))+0.25); ax2.set_xticklabels(DETS)
ax2.set_title("emissions avoided (common set)"); ax2.set_ylabel("kg CO₂e"); ax2.legend(title="policy")
plt.show()
print("Naive is wrong ≥70% under the weak D0 detector and 0% gated; where D2 makes the sensor "
      "reliable, gated captures ~1.9× the emissions of human-only at zero latency.")

## 5 · Sensitivity — the one knob, and where it bites

The only genuinely chosen constant is the **human-response horizon** `T_h`, which sets the blind
floor `c₁ = 1 − exp(−r_FA·T_h)`. Slide it and watch the coolant grant: it holds until roughly
`T_h ≈ 6–7 h` (the paper's derived operating point is ≈6.7 h), where `c₁` rises past coolant's
measured detectability and the grant demotes to blind — the framework telling you exactly which
deployment assumption that grant depends on.

In [ ]:
from contract.governance import derive_sigma_cutpoints

def horizon(T_h=4.0):
    cuts = derive_sigma_cutpoints(fa_rate=0.05, horizon_h=T_h)
    Ch = EpistemicContract(cuts=cuts)
    print(f"T_h = {T_h:.1f} h   →   c₁ = {cuts['c1']:.3f}   (c₂={cuts['c2']}, c₃={cuts['c3']})")
    for name in ("compressed_air_leak", "coolant_pump_fault"):
        g = Ch.standing_grant(REG[name], "defer_job", "D0")
        print(f"   {name:22s} D0 → σ={int(g.sigma)} ({SIGMA_NAME[int(g.sigma)]}), "
              f"{AUTONOMY_LABEL[g.autonomy].split(' ',1)[0]}")

interact(horizon, T_h=widgets.FloatSlider(min=1.0, max=12.0, step=0.1, value=4.0,
                                          description="T_h (h)"));

## 6 · Optional — *the LLM proposes, the contract disposes*

The PoC's agents are **deterministic** contract-net stand-ins, on purpose: the demonstration is
seeded and reproducible. A non-deterministic **LLM proposer** is the paper's explicit future-work
layer — and the whole point of the epistemic contract is that it sits *underneath* any proposer as
a deterministic verifier.

The cell below makes that concrete. An "eager" proposer suggests an autonomous action for a fault
where the sensor is **blind** (tool-wear under D0). Watch the contract **veto** it and escalate —
regardless of how confident the proposer was. Set an `OPENAI_API_KEY` to have a real LLM play the
proposer; otherwise a scripted eager proposer drives it so the cell always runs.

In [ ]:
import os, json

def llm_propose(situation_text):
    # Returns {'action','rationale'}. Uses a real LLM if OPENAI_API_KEY is set.
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        try:
            from openai import OpenAI
            client = OpenAI()
            msg = ("You are an energy-optimization agent in an MES. Given the situation, "
                   "propose ONE action from this list to cut carbon intensity: "
                   f"{list(ACTIONS.keys())}. Reply as JSON {{\"action\":..,\"rationale\":..}}.\n"
                   f"Situation: {situation_text}")
            r = client.chat.completions.create(model="gpt-4o-mini",
                    messages=[{"role":"user","content":msg}], temperature=0.2)
            return json.loads(r.choices[0].message.content)
        except Exception as e:
            print("(LLM call unavailable — using scripted proposer:", e, ")")
    return {"action": "defer_job",
            "rationale": "Grid CI is high; defer the batch to a cleaner window to cut emissions."}

# A fault the sensor is BLIND to under D0: tool wear (slow, sub-floor).
fc = REG["tool_wear"]; DET = "D0"
prop = llm_propose(f"tool-wear fault, severity {fc['severity_frac_nominal']}·Pb, "
                   f"slow onset ρ≈{fc['onset_ratio_nominal']}, detector {DET}")
print("LLM/agent PROPOSES :", prop["action"], "—", prop["rationale"])

gd = C.gate_action(prop["action"], s_hat=fc["severity_frac_nominal"],
                   rho_hat=fc["onset_ratio_nominal"], detector=DET)
print("CONTRACT DISPOSES  :", AUTONOMY_LABEL[gd.autonomy])
print("                     P_det(lo95)=%.3f → σ=%d (%s)" %
      (gd.p_det_lo95, int(gd.sigma), SIGMA_NAME[int(gd.sigma)]))
verdict = "VETOED → escalate to human" if gd.escalated else "permitted autonomously"
print("                    ", verdict)
print("\nThe proposer's confidence is irrelevant: the contract grants authority only from the "
      "sensing layer's *measured* ability to see this fault. Here it cannot, so it escalates.")

## What you just saw

The autonomy boundary is not a policy dial — it is a **measured property of the instrument**.
Drag a fault into the bright region and the agent may act; drag it into the dark region and the
same contract escalates to a human. Improving the *detector* (D0 → D2) is what legitimately grows
the autonomous region; loosening the *gate* never does. And any proposer — rules or LLM — is
disposed by the same measured boundary underneath it.

*You cannot decarbonize a factory you cannot see, and you cannot let an agent act where it cannot
see either.*